# nanoPLM quickstart

A minimal recipe for using an exported [nanoPLM](https://github.com/peymanvahidi/nanoplm)
protein language model: load it, tokenize sequences, compute embeddings, and
visualize a labeled dataset with UMAP. Only `transformers` and `torch` are
needed to load the model itself -- **no `nanoplm` install is required**.

Install this notebook's extra dependencies (dataset download and plotting):

```
pip install transformers torch umap-learn matplotlib datasets
```

## 1. Load the model

Change `REPO` below to your published model id (e.g. `"your-username/your-model"`),
or point it at a local export directory. `trust_remote_code=True` is required:
the export ships its own `configuration_nanoplm.py` / `modeling_nanoplm.py`.

In [ ]:
from transformers import AutoModel, AutoTokenizer
import torch

REPO = "acme/nanoplm-base"
tokenizer = AutoTokenizer.from_pretrained(REPO, trust_remote_code=True)
model = AutoModel.from_pretrained(REPO, trust_remote_code=True).eval()

sequences = ["MKALCLLLLPVLG", "MVLSPADKTNVKAAWGKV"]
batch = tokenizer(
    sequences, return_tensors="pt", padding=True, return_special_tokens_mask=True
)
special = batch.pop("special_tokens_mask").bool()
residue_mask = batch["attention_mask"].bool() & ~special

with torch.no_grad():
    out = model(**batch)

# per-residue embeddings, special tokens (BOS/EOS/register/padding) excluded
for i, seq in enumerate(sequences):
    per_residue = out.last_hidden_state[i][residue_mask[i]]
    per_sequence = per_residue.mean(dim=0)
    print(seq, per_residue.shape, per_sequence.shape)


In [ ]:
BATCH_SIZE = 16

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

model = model.to(device)

## 2. Tokenize

The tokenizer appends `</s>` (EOS) after every sequence. If this export was
trained with register tokens (`num_register_tokens > 0`), those follow the
EOS token. A beginning-of-sequence token (`<s>`) is optionally prepended when
the exported config sets `use_bos_token=True` (most nanoPLM checkpoints leave
it off). Amino-acid letters are case-insensitive -- lowercase input is
uppercased -- and the normalizer maps the ambiguous codes `UZOB` (U, Z, O, B)
to `X` before tokenization.

`return_special_tokens_mask=True` marks every one of these added tokens (plus
padding), which is how the embedding step below knows which positions are
real residues.

In [ ]:
demo_sequences = ["MKALCLLLLPVLG", "MVLSPADKTNVKAAWGKV"]
demo_batch = tokenizer(
    demo_sequences, return_tensors="pt", padding=True, return_special_tokens_mask=True
)
print(demo_batch["input_ids"])

## 3. Embed

Per-residue embeddings are `model(...).last_hidden_state`, one row per token.
A single per-sequence embedding is the mean over the *real residue* rows only
-- special tokens (BOS/EOS/register/padding) are excluded via the special
tokens mask, never by slicing to `len(seq)`: that slice is only correct when
no BOS is prepended and no register tokens are appended, which is not true of
every nanoPLM export.

Sequences longer than `MAX_SEQ_LEN` are truncated by the tokenizer so a
pathological input can't blow up memory; the public dataset loaded below is
filtered to the same bound instead, so every embedded residue there is real,
untruncated sequence.

In [ ]:
MAX_SEQ_LEN = 512  # cap on tokenized length; see markdown above


@torch.no_grad()
def embed(sequences, batch_size=BATCH_SIZE):
    """Mean-pooled per-sequence embeddings, special tokens excluded."""
    out = []
    for i in range(0, len(sequences), batch_size):
        chunk = sequences[i : i + batch_size]
        batch = tokenizer(
            chunk,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            return_special_tokens_mask=True,
        ).to(device)
        special = batch.pop("special_tokens_mask").bool()
        residue_mask = batch["attention_mask"].bool() & ~special
        hidden = model(**batch).last_hidden_state
        for j in range(len(chunk)):
            out.append(hidden[j][residue_mask[j]].mean(dim=0).float().cpu())
    return torch.stack(out)


demo_embeddings = embed(demo_sequences)
print(demo_embeddings.shape)

`model(..., output_hidden_states=True).hidden_states` exposes every layer's
output, in case you want an intermediate layer instead of the final one --
some downstream tasks embed better from a middle layer than from the last.

## 4. Analyze: UMAP

We visualize embeddings for a subcellular-localization dataset:
[`AI4Protein/DeepLocMulti`](https://huggingface.co/datasets/AI4Protein/DeepLocMulti)
(Apache-2.0 license), a re-split of the DeepLoc benchmark. Its `location`
column also carries a membrane-topology suffix (`S`/`M`/`U`); we use the
compartment before the first comma, DeepLoc's classic 10-way subcellular
label. nanoPLM's own eval pipeline runs the same DeepLoc-derived task as
biotrainer's `pbc_supervised` "scl" (subcellular localization) autoeval,
which trains a head on frozen embeddings -- so the cluster separation in the
plot below and that benchmark's numbers cross-reference the same signal.

The dataset is pinned to a fixed commit (`revision=`) for reproducibility,
sequences longer than `MAX_SEQ_LEN` are dropped (not truncated), and the
remainder is subsampled with a fixed seed to a few thousand sequences.

In [ ]:
from collections import Counter

from datasets import load_dataset

DATASET_REPO = "AI4Protein/DeepLocMulti"
DATASET_REVISION = "1af82ded911c8b3faadf185d9357193de9f301c9"
N_SAMPLES = 2000
SEED = 0

raw = load_dataset(DATASET_REPO, revision=DATASET_REVISION, split="train")
raw = raw.filter(lambda ex: len(ex["aa_seq"]) <= MAX_SEQ_LEN)
raw = raw.shuffle(seed=SEED).select(range(min(N_SAMPLES, len(raw))))

sequences = raw["aa_seq"]
# "location" also carries a membrane-topology suffix (S/M/U); the compartment
# before the first comma is DeepLoc's classic 10-way subcellular label.
labels = [loc.split(",")[0] for loc in raw["location"]]
print(len(sequences), "sequences")
print(Counter(labels))

In [ ]:
embeddings = embed(sequences)

import umap
import matplotlib.pyplot as plt

coords = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine").fit_transform(
    embeddings.numpy()
)

fig, ax = plt.subplots(figsize=(7, 6))
for label in sorted(set(labels)):
    mask = [l == label for l in labels]
    ax.scatter(coords[mask, 0], coords[mask, 1], s=8, label=label, alpha=0.7)
ax.legend(markerscale=2, fontsize=8)
ax.set_title("nanoPLM embeddings (UMAP)")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

**What to read from the plot:** points that cluster together have similar
mean-pooled embeddings; if clusters roughly line up with the color (the
localization label), the model's embeddings carry localization signal
without ever being trained on it.

**Caveats:** UMAP distances are not metric -- the distance *between* two
clusters, and cluster size/density, are not meaningful, only which points
group together. Visual cluster separation is not a benchmark: it's a sanity
check, not a substitute for the actual `pbc_supervised` numbers a trained
probe would give you.